In [ ]:
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install pyyaml

In [ ]:
!git clone https://github.com/nam13092003/NER-financial-data.git /kaggle/working/NER_finance

In [ ]:
import yaml

CONFIG_PATH = "/kaggle/working/NER_finance/configs/default.yaml"
MY_CONFIG   = "/kaggle/working/my_config.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# ── Change experiment settings here ──────────────────────────
cfg["training"]["format"]  = "instruction"  # causal | instruction
# cfg["model"]["name"]       = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"
cfg["model"]["name"]       = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit"
# cfg["model"]["name"]       = "unsloth/llama-3-8b-Instruct-bnb-4bit"
# ─────────────────────────────────────────────────────────────

# FIRE dataset files
cfg["data"]["train_files"] = [
    {"path": "/kaggle/input/fire-dataset/fire_train.json"},
]
cfg["data"]["eval_files"] = [
    {"path": "/kaggle/input/fire-dataset/fire_dev.json"},
]
cfg["data"]["test_files"] = [
    {"path": "/kaggle/input/fire-dataset/fire_test.json"},
]

cfg["training"]["batch_size"] = 8
cfg["training"]["epochs"] = 3
cfg["training"]["eval_steps"] = 50
cfg["training"]["save_steps"] = 50
cfg["training"]["eval_subset_size"] = 0.1     
cfg["training"]["early_stopping_patience"] = 5

with open(MY_CONFIG, "w") as f:
    yaml.dump(cfg, f, allow_unicode=True)

print("Config saved to:", MY_CONFIG)

In [ ]:
!accelerate config default

In [ ]:
CONFIG = "/kaggle/working/my_config.yaml"   # or use default.yaml

!accelerate launch \
    --multi_gpu \
    --num_processes=2 \
    /kaggle/working/NER_finance/train.py \
    --config {CONFIG} \
    2>&1 | tee /kaggle/working/training_log.txt

In [ ]:
# CONFIG = "/kaggle/working/my_config.yaml"   # or use default.yaml

# !accelerate launch \
#     --multi_gpu \
#     --num_processes=2 \
#     /kaggle/working/NER_finance/evaluate.py \
#     --config {CONFIG} \
#     --checkpoint /kaggle/working/outputs/final_lora_adapter \
#     --batch_size 10 \
#     --output /kaggle/working/predictions.jsonl \
#     2>&1 | tee /kaggle/working/eval_log.txt